# Notebook 4: Reflection and replanning

Two ideas that make agents more reliable:

- **Reflection**: the model reviews its own answer and improves it.
- **Replanning**: when a tool fails, the agent reads the error and tries again.

We will see both.

## Setup: run these first

Every notebook starts the same way: install the library, load your key, and
pick a model. Run the next three cells in order.

**Save yourself time:** store your API key once in Colab Secrets (the key icon on
the left sidebar). Add a secret named `ANTHROPIC_API_KEY`, paste your key as the
value, and turn on "Notebook access". After that, every notebook picks it up
automatically and you never paste it again.

In [ ]:
# 1) Install the Anthropic library (Colab forgets it between sessions, so we
#    install each time). The -q just keeps the output quiet.
!pip install -q anthropic

In [ ]:
# 2) Load your Anthropic API key.
#    Easiest: store it once in Colab Secrets (key icon on the left), named
#    ANTHROPIC_API_KEY, with "Notebook access" on. Then this reads it for you.
import anthropic

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None  # not on Colab, or no secret set yet

if not api_key:
    # Fallback: ask once if no secret is found. The input is hidden and not saved.
    from getpass import getpass
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

client = anthropic.Anthropic(api_key=api_key)
print("Your client is ready.")

In [ ]:
# 3) Choose the model. claude-opus-4-8 is the most capable. For faster, cheaper
#    runs while practising you can switch this to "claude-haiku-4-5".
MODEL = "claude-opus-4-8"
print("Using model:", MODEL)

## Part A: reflection (make, judge, improve)

First we ask for a quick draft. Then we ask the model to critique its own draft
and write a better version. This is the make, judge, improve loop in two steps.

In [ ]:
def ask(prompt):
    response = client.messages.create(
        model=MODEL, max_tokens=1024, messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in response.content if b.type == "text")

draft = ask("Write a 2 sentence summary of why planning helps an AI agent.")
print("DRAFT:\n", draft)

critique_prompt = (
    "Here is a draft:\n"
    + draft
    + "\nName one concrete way to improve it, then write the improved 2 sentence version."
)
improved = ask(critique_prompt)
print("\nIMPROVED:\n", improved)

## Part B: replanning when a tool fails

The weather tool below returns an error if the city has no country. Watch the
agent read that error and retry with a corrected request. We reuse the agent
loop, and we surface tool errors with `is_error` so the model knows it failed.

In [ ]:
# A weather tool that fails on its own when the city is ambiguous, so we can see
# the agent read the error and try again with a better input.
weather_data = {
    "geneva, switzerland": "18 degrees C and cloudy",
    "nairobi, kenya": "26 degrees C and sunny",
}

weather_tool = {
    "name": "get_weather",
    "description": "Get the weather for a city. Include the country, for example 'Geneva, Switzerland'.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City and country, for example 'Nairobi, Kenya'."}
        },
        "required": ["city"],
    },
}


def run_weather(city):
    key = city.lower().strip()
    if "," not in key:
        # The error message tells the model how to fix its request.
        return "Error: many cities share that name. Please include the country, like 'Geneva, Switzerland'."
    return weather_data.get(key, "Error: no data for that city. Try Geneva, Switzerland or Nairobi, Kenya.")

In [ ]:
tools = [weather_tool]

def run_agent(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages,
        )
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print("Thought:", block.text.strip())
        if response.stop_reason != "tool_use":
            for block in response.content:
                if block.type == "text":
                    return block.text
            return ""
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print("Action:", block.name, block.input)
                observation = run_weather(block.input["city"])
                print("Observation:", observation)
                is_error = observation.startswith("Error")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": observation,
                    "is_error": is_error,
                })
        messages.append({"role": "user", "content": tool_results})
    return "Stopped after the maximum number of steps."

In [ ]:
# Ask in a way that first triggers the error (no country), so we can watch the
# agent notice the problem and replan with a corrected city.
print(run_agent("What is the weather in Geneva?"))

## Recap

- Reflection turned a rough draft into a better answer.
- Replanning let the agent recover from a failed tool call instead of giving up.

## Your turn

Add a second city and country to `weather_data`, then ask about it. Try asking
about a city that is not in the data and see how the agent responds to the error.